<a href="https://colab.research.google.com/github/Nosindiso13/Hey_Farmer-/blob/main/cropyield_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

In [2]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
import numpy as np

# Load a pre-trained MobileNetV2 model for demonstration
pest_model = MobileNetV2(weights='imagenet')

def detect_pest(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)

    preds = pest_model.predict(x)
    return decode_predictions(preds, top=3)[0]

print("Pest detection model (MobileNetV2) loaded successfully.")

14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Pest detection model (MobileNetV2) loaded successfully.


In [3]:
import requests

# Streamlit public URL obtained from previous ngrok execution
streamlit_public_url = "https://libidinally-unconcordant-jose.ngrok-free.dev"

print(f"Checking Streamlit app at: {streamlit_public_url}")

try:
    response = requests.get(streamlit_public_url, timeout=10)
    response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)

    print(f"\nStreamlit app is running and reachable! Status Code: {response.status_code}")
    # Optionally, print first few lines of content to confirm it's an HTML page
    print("First 500 characters of response content:")
    print(response.text[:500])

except requests.exceptions.ConnectionError:
    print(f"Error: Could not connect to the Streamlit app at {streamlit_public_url}. The app might not be running or the ngrok tunnel is down.")
except requests.exceptions.Timeout:
    print(f"Error: Request to {streamlit_public_url} timed out. The Streamlit app might be slow to respond or not fully started.")
except requests.exceptions.RequestException as e:
    print(f"An HTTP error occurred: {e}")
    if response is not None:
        print(f"Response status code: {response.status_code}")
        print(f"Response body (first 500 chars): {response.text[:500]}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Checking Streamlit app at: https://libidinally-unconcordant-jose.ngrok-free.dev
An HTTP error occurred: 404 Client Error: Not Found for url: https://libidinally-unconcordant-jose.ngrok-free.dev/
Response status code: 404
Response body (first 500 chars): <!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
  


In [4]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
import os

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# __ Optional XGBoost ──────────────────────────────────────────────────────────
try:
    from xgboost import XGBRegressor
    USE_XGB = True
except ImportError:
    USE_XGB = False

# __ 1. Load & inspect data ────────────────────────────────────────────────────
print("=" * 60)
print("CROP YIELD PREDICTION — ML PIPELINE")
print("=" * 60)

# --- FIX: Removed direct download due to persistent 404 errors ---
# Please upload 'yield_df.csv' directly to your Colab environment (e.g., via the files tab)
# If you don't have the file, you can search for 'yield_df.csv' online or on Kaggle.

# Load the dataset from local file
try:
    df = pd.read_csv('yield_df.csv', index_col=0)
except FileNotFoundError:
    print("Error: 'yield_df.csv' not found. Please upload the file to your Colab environment.")
    raise

if df.empty:
    print("Error: DataFrame 'df' is empty after reading CSV. The uploaded file might be malformed.")
    raise ValueError("Exiting due to empty DataFrame, cannot proceed with analysis.")
# --- End of fix for data loading ---

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nTarget (hg/ha_yield) stats:")
print(df['hg/ha_yield'].describe().round(1))
# __ 2. Feature engineering ────────────────────────────────────────────────────
# Log-transform target to handle skew
df['log_yield'] = np.log1p(df['hg/ha_yield'])

# Rename for convenience
df = df.rename(columns={
    'year': 'Year',
    'avg_rainfall': 'rainfall',
    'Pestcides': 'pesticides',
    'avg_Temp': 'temperature'
})

CATEGORICAL = ['Area', 'Item']
NUMERICAL   = ['Year', 'rainfall', 'pesticides', 'temperature']
TARGET      = 'log_yield'

X = df[CATEGORICAL + NUMERICAL]
y = df[TARGET]

# __ 3. Preprocessing pipeline ─────────────────────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL),
    ('num', StandardScaler(), NUMERICAL),
])

# __ 4. Define models ──────────────────────────────────────────────────────────
models = {
    'Random Forest':        RandomForestRegressor(n_estimators=200, max_depth=15,
                                                  min_samples_leaf=2, n_jobs=-1,
                                                  random_state=42),
    'Gradient Boosting':    GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                                      learning_rate=0.1,
                                                      subsample=0.8, random_state=42),
    'Ridge Regression':     Ridge(alpha=10.0),
    'KNN Regressor':        KNeighborsRegressor(n_neighbors=10, weights='distance',
                                                n_jobs=-1),
}

if USE_XGB:
    models['XGBoost'] = XGBRegressor(n_estimators=300, max_depth=6,
                                     learning_rate=0.05, subsample=0.8,
                                     colsample_bytree=0.8, random_state=42,
                                     tree_method='hist')

# __ 5. Train / test split ──────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTrain size: {len(X_train):,}  |  Test size: {len(X_test):,}")

# __ 6. Train, evaluate, cross-validate ────────────────────────────────────────
print("\n" + "=" * 60)
print("MODEL EVALUATION (log-scale RMSE, R², MAE + 5-fold CV R²)")
print("=" * 60)

results = {}

for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])

    # Fit
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    # Metrics on log scale
    rmse_log = np.sqrt(mean_squared_error(y_test, y_pred))
    r2        = r2_score(y_test, y_pred)
    mae_log   = mean_absolute_error(y_test, y_pred)

    # Back-transform for interpretable MAE (hg/ha)
    mae_orig = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred))

    # 5-fold cross-validation R²
    cv_scores = cross_val_score(pipe, X, y, cv=5, scoring='r2', n_jobs=-1)

    results[name] = {
        'pipe':     pipe,
        'y_pred':   y_pred,
        'rmse_log': rmse_log,
        'r2':       r2,
        'mae_log':  mae_log,
        'mae_orig': mae_orig,
        'cv_mean':  cv_scores.mean(),
        'cv_std':   cv_scores.std(),
    }

    print(f"\n{name}")
    print(f"  RMSE (log):       {rmse_log:.4f}")
    print(f"  R²:               {r2:.4f}")
    print(f"  MAE (hg/ha):      {mae_orig:,.0f}")
    print(f"  CV R² (5-fold):   {cv_scores.mean():.4f} \u00b1 {cv_scores.std():.4f}")

# Best model
best_name = max(results, key=lambda k: results[k]['cv_mean'])
print(f"\n\u2605  Best model by CV R²: {best_name} ({results[best_name]['cv_mean']:.4f})")

# __ 7. Feature importance (Random Forest) ─────────────────────────────────────
rf_pipe  = results['Random Forest']['pipe']
rf_model = rf_pipe.named_steps['model']
ohe_cols = rf_pipe.named_steps['prep'].transformers_[0][1].get_feature_names_out(CATEGORICAL)
feature_names = list(ohe_cols) + NUMERICAL

importances = pd.Series(rf_model.feature_importances_, index=feature_names)

# Aggregate OHE columns back to Area / Item
def agg_importance(imp, prefix):
    mask = imp.index.str.startswith(prefix)
    return imp[mask].sum()

agg = pd.Series({
    'Area':        agg_importance(importances, 'Area'),
    'Item (crop)': agg_importance(importances, 'Item'),
    'Year':        importances['Year'],
    'Rainfall':    importances['rainfall'],
    'Pesticides':  importances['pesticides'],
    'Temperature': importances['temperature'],
}).sort_values(ascending=True)

# __ 8. Plots ───────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
fig.patch.set_facecolor('#F8F7F4')
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

PALETTE = {
    'Random Forest':     '#1D9E75',
    'Gradient Boosting': '#534AB7',
    'Ridge Regression':  '#185FA5',
    'KNN Regressor':     '#BA7517',
    'XGBoost':           '#D85A30',
}

# __ 8a. CV R² comparison bar chart ────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
names  = list(results.keys())
cv_means = [results[n]['cv_mean'] for n in names]
cv_stds  = [results[n]['cv_std']  for n in names]
colors   = [PALETTE.get(n, '#888') for n in names]

bars = ax1.barh(names, cv_means, xerr=cv_stds, color=colors, height=0.55,
                error_kw=dict(elinewidth=1.2, capsize=4, ecolor='#555'))
ax1.set_xlabel('CV R² score', fontsize=10)
ax1.set_title('Cross-validated R²\n(5-fold, higher = better)', fontsize=11, fontweight='500')
ax1.set_xlim(0, 1.05)
ax1.axvline(0.9, color='#aaa', lw=0.8, ls='--', alpha=0.6)
for bar, val in zip(bars, cv_means):
    ax1.text(val + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)
ax1.set_facecolor('#F8F7F4')
ax1.spines[['top','right']].set_visible(False)

# __ 8b. RMSE comparison ────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
rmses = [results[n]['rmse_log'] for n in names]
bars2 = ax2.barh(names, rmses, color=colors, height=0.55)
ax2.set_xlabel('RMSE (log scale)', fontsize=10)
ax2.set_title('RMSE on test set\n(lower = better)', fontsize=11, fontweight='500')
for bar, val in zip(bars2, rmses):
    ax2.text(val + 0.002, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)
ax2.set_facecolor('#F8F7F4')
ax2.spines[['top','right']].set_visible(False)

# __ 8c. Feature importance (RF) ───────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
feat_colors = ['#1D9E75' if v > 0.15 else '#9FE1CB' for v in agg.values]
ax3.barh(agg.index, agg.values, color=feat_colors, height=0.55)
ax3.set_xlabel('Importance', fontsize=10)
ax3.set_title('Feature importance\n(Random Forest, aggregated)', fontsize=11, fontweight='500')
ax3.set_facecolor('#F8F7F4')
ax3.spines[['top','right']].set_visible(False)

# __ 8d–f. Actual vs predicted (top 3 models by CV R²) ─────────────────────────
top3 = sorted(results, key=lambda k: results[k]['cv_mean'], reverse=True)[:3]

for i, name in enumerate(top3):
    ax = fig.add_subplot(gs[1, i])
    y_t  = np.expm1(y_test)
    y_p  = np.expm1(results[name]['y_pred'])
    r2v  = results[name]['r2']

    ax.scatter(y_t, y_p, alpha=0.15, s=6, color=PALETTE.get(name, '#888'),
               rasterized=True)
    lim = max(y_t.max(), y_p.max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', lw=1, alpha=0.5, label='Perfect fit')
    ax.set_xlabel('Actual yield (hg/ha)', fontsize=9)
    ax.set_ylabel('Predicted yield (hg/ha)', fontsize=9)
    ax.set_title(f'{name}\nR² = {r2v:.4f}', fontsize=10, fontweight='500')
    ax.set_facecolor('#F8F7F4')
    ax.spines[['top','right']].set_visible(False)
    ax.ticklabel_format(style='sci', axis='both', scilimits=(0,0))

fig.suptitle('Crop Yield Prediction — Model Evaluation', fontsize=14,
             fontweight='600', y=1.01)

# Create the directory if it doesn't exist
os.makedirs('/mnt/user-data/outputs/', exist_ok=True)
plt.savefig('/mnt/user-data/outputs/crop_yield_model_evaluation.png',
            dpi=150, bbox_inches='tight', facecolor='#F8F7F4')
print("\nPlot saved.")

# __ 9. Summary table ───────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SUMMARY TABLE")
print("=" * 60)
summary = pd.DataFrame({
    'Model':       names,
    'CV R²':       [f"{results[n]['cv_mean']:.4f} \u00b1 {results[n]['cv_std']:.4f}" for n in names],
    'Test R²':     [f"{results[n]['r2']:.4f}"       for n in names],
    'RMSE (log)':  [f"{results[n]['rmse_log']:.4f}" for n in names],
    'MAE (hg/ha)': [f"{results[n]['mae_orig']:,.0f}" for n in names],
}).set_index('Model')
print(summary.to_string())

print(f"\n\u2605  Best model: {best_name}")
print("\nDone.")

CROP YIELD PREDICTION — ML PIPELINE

Dataset shape: (367, 7)
Columns: ['Area', 'Item', 'year', 'hg/ha_yield', 'avg_rainfall', 'Pestcides', 'avg_Temp']

Target (hg/ha_yield) stats:
count       367.0
mean      39784.5
std       42710.9
min        2046.0
25%       12914.5
50%       22133.0
75%       54000.0
max      172628.0
Name: hg/ha_yield, dtype: float64

Train size: 293  |  Test size: 74

MODEL EVALUATION (log-scale RMSE, R², MAE + 5-fold CV R²)

Random Forest
  RMSE (log):       0.2756
  R²:               0.9296
  MAE (hg/ha):      5,526
  CV R² (5-fold):   0.8714 ± 0.0373

Gradient Boosting
  RMSE (log):       0.2384
  R²:               0.9473
  MAE (hg/ha):      5,388
  CV R² (5-fold):   0.8626 ± 0.0473

Ridge Regression
  RMSE (log):       0.4140
  R²:               0.8411
  MAE (hg/ha):      10,061
  CV R² (5-fold):   0.7968 ± 0.0509

KNN Regressor
  RMSE (log):       0.4976
  R²:               0.7704
  MAE (hg/ha):      10,650
  CV R² (5-fold):   0.7911 ± 0.0820

XGBoost
  RMSE

In [5]:
import pandas as pd

try:
    df_check = pd.read_csv('yield_df.csv', index_col=0)
    print("Successfully loaded 'yield_df.csv'.")
    print("\nFirst 5 rows of the DataFrame:")
    display(df_check.head())
    print("\nDataFrame Information (column types and non-null counts):")
    df_check.info()
    print("\nDescriptive Statistics:")
    display(df_check.describe())
except FileNotFoundError:
    print("Error: 'yield_df.csv' not found. Please ensure the file is uploaded to your Colab environment.")
except Exception as e:
    print(f"An error occurred while reading the CSV: {e}")

Successfully loaded 'yield_df.csv'.

First 5 rows of the DataFrame:


,Area,Item,year,hg/ha_yield,avg_rainfall,Pestcides,avg_Temp
0,Zambia,Maize,2026,14316,1020,1080.0,21.43
1,Zambia,Potatoes,2026,90000,1020,1080.0,21.43
2,Zambia,"Rice, paddy",2026,9664,1020,1080.0,21.43
3,Zambia,Sorghum,2026,4042,1020,1080.0,21.43
4,Zambia,Soybeans,2026,8986,1020,1080.0,21.43



DataFrame Information (column types and non-null counts):
<class 'pandas.core.frame.DataFrame'>
Index: 367 entries, 0 to 366
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Area          367 non-null    object 
 1   Item          367 non-null    object 
 2   year          367 non-null    int64  
 3   hg/ha_yield   367 non-null    int64  
 4   avg_rainfall  367 non-null    int64  
 5   Pestcides     367 non-null    float64
 6   avg_Temp      367 non-null    float64
dtypes: float64(2), int64(3), object(2)
memory usage: 22.9+ KB

Descriptive Statistics:


,year,hg/ha_yield,avg_rainfall,Pestcides,avg_Temp
count,367.0,367.000000,367.000000,367.000000,367.000000
mean,2026.0,39784.468665,838.005450,2351.240109,20.946948
std,0.0,42710.853585,181.747107,1147.340073,0.468439
min,2026.0,2046.000000,657.000000,716.000000,19.760000
25%,2026.0,12914.500000,657.000000,1670.000000,20.660000
50%,2026.0,22133.000000,657.000000,1670.000000,20.870000
75%,2026.0,54000.000000,1020.000000,3094.080000,21.200000
max,2026.0,172628.000000,1020.000000,6753.000000,21.970000


In [6]:
import joblib
import os

print("Joblib and OS modules imported.")

Joblib and OS modules imported.


In [7]:
import os
import joblib

# Ensure the exact directory expected by main.py exists
output_dir = 'model_artifacts'
os.makedirs(output_dir, exist_ok=True)

# Save the XGBoost pipeline to the correct location
model_path = os.path.join(output_dir, 'xgboost_pipeline.joblib')
try:
    # Assuming 'results' is available from cell bF695J1hEgBB
    joblib.dump(results['XGBoost']['pipe'], model_path)
    print(f"Model verified and saved to: {model_path}")
except NameError:
    print("Error: 'results' dictionary not found. Please ensure the training cell (bF695J1hEgBB) has been executed.")

Model verified and saved to: model_artifacts/xgboost_pipeline.joblib


In [8]:
print('''import streamlit as st
import requests # Will be removed later, kept for initial modification clarity
import json
import pandas as pd
import numpy as np
import pickle # Replaced joblib with pickle
import os
import io
from PIL import Image
import asyncio # Added for async operations

# Imports for Gemini API
import google.genai as genai # Corrected import
from google.colab import userdata

# Imports for Pest Detection
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image

# --- Configuration & Global Variables ---
MODEL_PATH = 'model_artifacts/xgboost_pipeline.joblib'

# --- Caching Models for Efficiency ---
@st.cache_resource
def load_crop_yield_model():
    if not os.path.exists(MODEL_PATH):
        st.error(f"Crop Yield Model file not found at {MODEL_PATH}.")
        return None
    try:
        with open(MODEL_PATH, 'rb') as f:
            pipeline = pickle.load(f) # Replaced joblib.load with pickle.load
        return pipeline
    except Exception as e:
        st.error(f"Error loading Crop Yield model: {e}")
        return None

@st.cache_resource
def load_pest_detection_model():
    try:
        model = MobileNetV2(weights='imagenet')
        return model
    except Exception as e:
        st.error(f"Error loading Pest Detection model: {e}")
        return None

@st.cache_resource
def load_gemini_model():
    try:
        # Ensure API key is configured
        api_key = userdata.get('crop_key')
        if not api_key:
            st.error("Google API Key not found. Please set 'crop_key' in Colab secrets.")
            return None
        # Pass API key directly to the GenerativeModel constructor
        model = genai.GenerativeModel('gemini-pro', api_key=api_key)
        return model
    except Exception as e:
        st.error(f"Error initializing Gemini model: {e}")
        return None

# --- Helper Functions (moved from main.py) ---
def predict_yield_helper(input_data_df: pd.DataFrame, model_pipeline_local) -> list[float]:
    if model_pipeline_local is None:
        st.error("Crop Yield Model is not loaded.")
        return []

    # Ensure column order matches training data
    expected_columns = ['Area', 'Item', 'Year', 'rainfall', 'pesticides', 'temperature']
    df_processed = input_data_df[expected_columns]

    log_predictions = model_pipeline_local.predict(df_processed)
    original_scale_predictions = np.expm1(log_predictions)
    return original_scale_predictions.tolist()

def detect_pest_helper(image_bytes: bytes, pest_model_local) -> list[dict]:
    if pest_model_local is None:
        st.error("Pest Detection Model is not loaded.")
        return []
    try:
        img = Image.open(io.BytesIO(image_bytes)).resize((224, 224))
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)

        preds = pest_model_local.predict(x)
        decoded_preds = decode_predictions(preds, top=3)[0]

        results = [{"label": label, "description": description, "probability": float(prob)}
                   for (_, label, prob) in decoded_preds]
        return results
    except Exception as e:
        st.error(f"Image processing or pest detection failed: {e}")
        return []

async def get_gemini_response(message: str, gemini_model_local) -> str:
    if gemini_model_local is None:
        st.error("Gemini model is not initialized.")
        return "Error: Gemini model not available."
    try:
        response = await gemini_model_local.generate_content_async(message)
        return response.text
    except Exception as e:
        st.error(f"Gemini API call failed: {e}")
        return "Error: Could not get a response from AI."

# --- Load all models at startup ---
crop_yield_pipeline = load_crop_yield_model()
pest_detection_model = load_pest_detection_model()
gemini_llm = load_gemini_model()

# --- Streamlit App Layout ---
st.set_page_config(page_title="Crop Yield Predictor & Advisory", layout="centered")

st.title("Crop Yield Prediction & Farmer Advisory")
st.markdown("Use the sections below to predict crop yields, detect pests, and get AI-powered advice.")

# Create tabs
tab1, tab2, tab3 = st.tabs(["Crop Yield Prediction", "Pest Detection", "Farmer Advisory"])

with tab1:
    st.subheader("Input Parameters for Yield Prediction")

    AREAS = ['Zambia', 'Zimbabwe']
    ITEMS = ['Wheat', 'Maize', 'Potatoes', 'Rice, paddy', 'Sorghum', 'Soybeans', 'Cassava', 'Sweet potatoes']
    MIN_YEAR = 2020
    MAX_YEAR = 2030
    MIN_RAINFALL = 0.0
    MAX_RAINFALL = 3500.0
    MIN_PESTICIDES = 0.0
    MAX_PESTICIDES = 150000.0
    MIN_TEMP = -20.0
    MAX_TEMP = 40.0

    with st.form("prediction_form"):
        col1, col2 = st.columns(2)
        with col1:
            area = st.selectbox("Area", options=AREAS, index=0)
            item = st.selectbox("Item (Crop Type)", options=ITEMS, index=0)
            year = st.number_input("Year", min_value=MIN_YEAR, max_value=MAX_YEAR, value=2022, step=1)
        with col2:
            rainfall = st.number_input("Average Annual Rainfall (mm)", min_value=MIN_RAINFALL, max_value=MAX_RAINFALL, value=1200.0, step=10.0, format="%.1f")
            pesticides = st.number_input("Pesticides (tonnes)", min_value=MIN_PESTICIDES, max_value=MAX_PESTICIDES, value=60000.0, step=100.0, format="%.1f")
            temperature = st.number_input("Average Temperature (°C)", min_value=MIN_TEMP, max_value=MAX_TEMP, value=25.5, step=0.1, format="%.1f")

        submitted = st.form_submit_button("Get Prediction")

        if submitted:
            input_data = {
                "Area": area,
                "Item": item,
                "Year": year,
                "rainfall": rainfall,
                "pesticides": pesticides,
                "temperature": temperature
            }
            input_df = pd.DataFrame([input_data])

            if crop_yield_pipeline:
                predicted_yields = predict_yield_helper(input_df, crop_yield_pipeline)
                if predicted_yields:
                    predicted_yield = predicted_yields[0]
                    st.success(f"Predicted Crop Yield: **{predicted_yield:,.0f} hg/ha**")
                    st.balloons()
                else:
                    st.error("Prediction could not be made.")
            else:
                st.warning("Crop Yield Model not available. Please check the logs.")

with tab2:
    st.subheader("Pest Detection")
    st.write("Upload an image of a crop or pest to get a detection from our AI model.")

    uploaded_file = st.file_uploader("Choose an image...", type=["jpg", "jpeg", "png"])

    if uploaded_file is not None:
        st.image(uploaded_file, caption="Uploaded Image", use_column_width=True)
        st.write("")
        st.write("Detecting pest...")

        image_bytes = uploaded_file.getvalue()

        if pest_detection_model:
            detections = detect_pest_helper(image_bytes, pest_detection_model)
            if detections:
                st.success("Detection Results:")
                for detection in detections:
                    st.write(f"- **{detection['description']}** (Confidence: {detection['probability']:.2f})")
            else:
                st.info("No significant detections found.")
        else:
            st.warning("Pest Detection Model not available. Please check the logs.")

with tab3:
    st.subheader("Farmer Advisory Chat")
    st.markdown("--- Nosindiso AI Demo --- ")

    if "messages" not in st.session_state:
        st.session_state.messages = []

    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])

    if prompt := st.chat_input("How can I help you, farmer?"):
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.markdown(prompt)

        with st.spinner("Getting advice from the AI..."):
            if gemini_llm:
                response_text = st.session_state.get('gemini_response_cache', {})
                if prompt not in response_text:
                    response_text[prompt] = asyncio.run(get_gemini_response(prompt, gemini_llm))
                    st.session_state['gemini_response_cache'] = response_text

                ai_response = response_text.get(prompt, "Error: Could not get a response from AI.")
                st.session_state.messages.append({"role": "assistant", "content": ai_response})
                with st.chat_message("assistant"):
                    st.markdown(ai_response)
            else:
                error_msg = "AI chat model not available. Please check API key configuration."
                st.error(error_msg)
                st.session_state.messages.append({"role": "assistant", "content": error_msg})
                with st.chat_message("assistant"):
                    st.markdown(error_msg)''')


import streamlit as st
import requests # Will be removed later, kept for initial modification clarity
import json
import pandas as pd
import numpy as np
import pickle # Replaced joblib with pickle
import os
import io
from PIL import Image
import asyncio # Added for async operations

# Imports for Gemini API
import google.genai as genai # Corrected import
from google.colab import userdata

# Imports for Pest Detection
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image

# --- Configuration & Global Variables ---
MODEL_PATH = 'model_artifacts/xgboost_pipeline.joblib'

# --- Caching Models for Efficiency ---
@st.cache_resource
def load_crop_yield_model():
    if not os.path.exists(MODEL_PATH):
        st.error(f"Crop Yield Model file not found at {MODEL_PATH}.")
        return None
    try:
        with open(MODEL_PATH, 'rb'

In [10]:
with open('streamlit_app.py', 'r') as f:
    streamlit_code = f.read()
print(streamlit_code)

FileNotFoundError: [Errno 2] No such file or directory: 'streamlit_app.py'

In [11]:
import joblib
import pandas as pd
import numpy as np

def predict_yield(input_data_df: pd.DataFrame) -> pd.Series:
    """
    Predicts crop yield (hg/ha) for new input data using the saved XGBoost model pipeline.

    Args:
        input_data_df (pd.DataFrame): A DataFrame containing new data with columns
                                      'Area', 'Item', 'Year', 'rainfall', 'pesticides',
                                      and 'temperature'.

    Returns:
        pd.Series: Predicted crop yields in hg/ha (original scale).
    """
    model_path = 'model_artifacts/xgboost_pipeline.joblib'
    try:
        # Load the saved pipeline
        pipeline = joblib.load(model_path)
    except FileNotFoundError:
        print(f"Error: Model pipeline not found at {model_path}. Please ensure it is saved.")
        return pd.Series()
    except Exception as e:
        print(f"An error occurred while loading the model: {e}")
        return pd.Series()

    # Make predictions (pipeline handles preprocessing)
    log_predictions = pipeline.predict(input_data_df)

    # Inverse transform to get predictions in original scale
    original_scale_predictions = np.expm1(log_predictions)

    return pd.Series(original_scale_predictions, index=input_data_df.index)

# --- Test the prediction function with sample data ---
print("\n--- Testing the predict_yield function ---")

# Create a sample DataFrame for prediction
sample_data = pd.DataFrame({
    'Area': ['India', 'Brazil', 'France', 'India'],
    'Item': ['Wheat', 'Maize', 'Potatoes', 'Rice, paddy'],
    'Year': [2022, 2023, 2022, 2021],
    'rainfall': [1200.0, 1800.0, 700.0, 1100.0],
    'pesticides': [60000.0, 80000.0, 30000.0, 55000.0],
    'temperature': [25.5, 28.0, 15.0, 26.5]
})

print("Sample Input Data:")
print(sample_data)

# Get predictions
predicted_yields = predict_yield(sample_data)

print("\nPredicted Crop Yields (hg/ha):")
print(predicted_yields.apply(lambda x: f'{x:,.0f}'))
print("\n--- Prediction function tested successfully ---")



--- Testing the predict_yield function ---
Sample Input Data:
     Area         Item  Year  rainfall  pesticides  temperature
0   India        Wheat  2022    1200.0     60000.0         25.5
1  Brazil        Maize  2023    1800.0     80000.0         28.0
2  France     Potatoes  2022     700.0     30000.0         15.0
3   India  Rice, paddy  2021    1100.0     55000.0         26.5

Predicted Crop Yields (hg/ha):
0     62,599
1     15,130
2    137,247
3     24,184
dtype: object

--- Prediction function tested successfully ---


In [12]:
# First, install Streamlit if you haven't already
!pip install streamlit

import streamlit as st
import requests
import json
import pandas as pd
import numpy as np

# --- Configuration --- #
# Adjust this URL if your FastAPI is running on a different host or port,
# or if you are using a tool like ngrok to expose it.
# For local Colab environment, it's usually http://127.0.0.1:8000
API_BASE_URL = "http://127.0.0.1:8000"
PREDICT_API_URL = f"{API_BASE_URL}/predict"
CHAT_API_URL = f"{API_BASE_URL}/chat"

# --- Streamlit App --- #
st.set_page_config(page_title="Crop Yield Predictor & Advisory", layout="centered")

st.title("Crop Yield Prediction & Farmer Advisory")
st.markdown("Use the sections below to predict crop yields and get AI-powered advice.")

st.subheader("Input Parameters for Yield Prediction")

# Get unique values for Area and Item from the original DataFrame `df`
# These should ideally be consistent with the training data.
# In a real deployment, these lists would be pre-loaded or fetched from a config.

# Hardcoding based on the user's updated data for 'Area'
AREAS = ['Zambia', 'Zimbabwe']
ITEMS = ['Wheat', 'Maize', 'Potatoes', 'Rice, paddy', 'Sorghum', 'Soybeans', 'Cassava', 'Sweet potatoes']

# MIN/MAX values are derived from the original dataset and are generally stable
MIN_YEAR = 2020
MAX_YEAR = 2030
MIN_RAINFALL = 0.0
MAX_RAINFALL = 3500.0
MIN_PESTICIDES = 0.0
MAX_PESTICIDES = 150000.0
MIN_TEMP = -20.0
MAX_TEMP = 40.0


with st.form("prediction_form"):
    col1, col2 = st.columns(2)
    with col1:
        area = st.selectbox("Area", options=AREAS, index=0)
        item = st.selectbox("Item (Crop Type)", options=ITEMS, index=0)
        year = st.number_input("Year", min_value=MIN_YEAR, max_value=MAX_YEAR, value=2022, step=1)
    with col2:
        rainfall = st.number_input("Average Annual Rainfall (mm)", min_value=MIN_RAINFALL, max_value=MAX_RAINFALL, value=1200.0, step=10.0, format="%.1f")
        pesticides = st.number_input("Pesticides (tonnes)", min_value=MIN_PESTICIDES, max_value=MAX_PESTICIDES, value=60000.0, step=100.0, format="%.1f")
        temperature = st.number_input("Average Temperature (°C)", min_value=MIN_TEMP, max_value=MAX_TEMP, value=25.5, step=0.1, format="%.1f")

    submitted = st.form_submit_button("Get Prediction")

    if submitted:
        input_data = {
            "Area": area,
            "Item": item,
            "Year": year,
            "rainfall": rainfall,
            "pesticides": pesticides,
            "temperature": temperature
        }

        # API expects a list of CropYieldRequest objects
        payload = [input_data]

        try:
            response = requests.post(PREDICT_API_URL, headers={'Content-Type': 'application/json'}, data=json.dumps(payload))
            response.raise_for_status() # Raise an exception for HTTP errors

            predictions = response.json()
            predicted_yield = predictions['predictions'][0] # Assuming a single prediction

            st.success(f"Predicted Crop Yield: **{predicted_yield:,.0f} hg/ha**")
            st.balloons()

        except requests.exceptions.ConnectionError:
            st.error(f"Error: Could not connect to the API at {PREDICT_API_URL}. Please ensure your FastAPI server is running.")
        except requests.exceptions.RequestException as e:
            st.error(f"An error occurred during the API request: {e}")
            if response is not None:
                st.error(f"API Response Status: {response.status_code}")
                st.error(f"API Response Body: {response.text}")
        except json.JSONDecodeError:
            st.error("Error: Could not decode JSON from API response.")
            if response is not None:
                st.error(f"API Response Body: {response.text}")
        except Exception as e:
            st.error(f"An unexpected error occurred: {e}")

st.markdown("--- Nosindiso AI Demo --- ")

st.subheader("Farmer Advisory Chat")

# Initialize chat history
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat messages from history on app rerun
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# React to user input
if prompt := st.chat_input("How can I help you, farmer?"):
    # Display user message in chat message container
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Display assistant response in chat message container
    with st.spinner("Getting advice from the AI..."):
        try:
            chat_payload = {"message": prompt}
            chat_response = requests.post(
                CHAT_API_URL,
                headers={"Content-Type": "application/json"},
                data=json.dumps(chat_payload)
            )
            chat_response.raise_for_status() # Raise an exception for HTTP errors

            ai_response = chat_response.json().get("response", "Error: No response from AI.")
            st.session_state.messages.append({"role": "assistant", "content": ai_response})
            with st.chat_message("assistant"):
                st.markdown(ai_response)
        except requests.exceptions.ConnectionError:
            error_msg = f"Error: Could not connect to the API at {CHAT_API_URL}. Please ensure your FastAPI server is running and accessible."
            st.error(error_msg)
            st.session_state.messages.append({"role": "assistant", "content": error_msg})
            with st.chat_message("assistant"):
                st.markdown(error_msg)
        except requests.exceptions.RequestException as e:
            error_msg = f"An error occurred during the chat API request: {e}"
            st.error(error_msg)
            if chat_response is not None:
                st.error(f"API Response Status: {chat_response.status_code}")
                st.error(f"API Response Body: {chat_response.text}")
            st.session_state.messages.append({"role": "assistant", "content": error_msg})
            with st.chat_message("assistant"):
                st.markdown(error_msg)
        except json.JSONDecodeError:
            error_msg = "Error: Could not decode JSON from chat API response."
            st.error(error_msg)
            if chat_response is not None:
                st.error(f"API Response Body: {chat_response.text}")
            st.session_state.messages.append({"role": "assistant", "content": error_msg})
            with st.chat_message("assistant"):
                st.markdown(error_msg)
        except Exception as e:
            error_msg = f"An unexpected error occurred during chat: {e}"
            st.error(error_msg)
            st.session_state.messages.append({"role": "assistant", "content": error_msg})
            with st.chat_message("assistant"):
                st.markdown(error_msg)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 129.1 MB/s eta 0:00:00


2026-03-31 23:26:35.507 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 23:26:35.507 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 23:26:35.802 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-03-31 23:26:35.802 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 23:26:35.803 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 23:26:35.805 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 23:26:35.807 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [13]:
# Install the Google Generative AI library
!pip install -q -U google-generativeai

# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('yield_key')
genai.configure(api_key=GOOGLE_API_KEY)

In [14]:
# Initialize the Gemini API
# We'll use a suitable model for chat, for example, 'gemini-pro'
gemini_model = genai.GenerativeModel('gemini-pro')

In [20]:
pip install google genai

  Using cached genai-2.1.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached ipython-8.39.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached openai-0.27.10-py3-none-any.whl.metadata (13 kB)
  Using cached tiktoken-0.3.3.tar.gz (25 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached jedi-0.19.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached stack_data-0.6.3-py3-none-any.whl.metadata (18 kB)
  Using cached traitlets-5.14.3-py3-none-any.whl.metadata (10 kB)
  Using cached executing-2.2.1-py2.py3-none-any.whl.metadata (8.9 kB)
  Using cached asttokens-3.0.1-py3-none-any.whl.metadata (4.9 kB)
  Using cached pure_eval-0.2.3-py3-none-any.whl.metadata (6.3 kB)
Using cached genai-2.1.0-py3-none-any.whl (16 kB)
Using cached ipython-8.39.0-py3-none-any.whl (831 kB)
Using cached openai-0.27.10-py3-none-any.whl (76 kB)
Using cached jedi-0.19.2-py2.py3-none-any.whl (1.6 MB)
Using cached trait

In [18]:
# Install build-essential which provides necessary compilers and build tools
print("Installing build-essential...")
!sudo apt-get update && sudo apt-get install -y build-essential
print("build-essential installation attempted.")

Installing build-essential...
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,475 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,954 kB]
Get:13 http://security.ubuntu.com/ubu

In [19]:
# Reinstall google-generativeai and tiktoken to ensure they build correctly with the new tools
# Using --upgrade to ensure a fresh install attempt
print("Reinstalling google-generativeai and tiktoken...")
!pip install --upgrade google-generativeai tiktoken
print("Reinstallation of google-generativeai and tiktoken attempted.")

Reinstalling google-generativeai and tiktoken...
Reinstallation of google-generativeai and tiktoken attempted.


In [16]:
!pip install tiktoken --no-build-isolation

In [21]:
pip install google-genai

In [22]:
%%writefile main.py
from pydantic import BaseModel, Field
from typing import List, Dict
import pandas as pd
import numpy as np
import joblib
import os
import io
from PIL import Image

# Import for Gemini API
import google.generativeai as genai

# New Tensorflow imports for pest detection
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image

# --- 0. Configure Gemini ---
# In a separate process, we use environment variables instead of userdata.get()
GOOGLE_API_KEY = os.getenv('YIELD_API_KEY')
if GOOGLE_API_KEY:
    genai.configure(api_key=GOOGLE_API_KEY)

# --- 1. Pydantic Models ---
class CropYieldRequest(BaseModel):
    Area: str
    Item: str
    Year: int = Field(..., gt=0)
    rainfall: float = Field(..., ge=0.0)
    pesticides: float = Field(..., ge=0.0)
    temperature: float = Field(..., ge=-50.0, le=70.0)

class ChatRequest(BaseModel):
    message: str

# --- 2.
app = (title="Crop Yield & Farmer Advisory API")

# --- 3. Global Models ---
model_pipeline = None
gemini_model = None
pest_model = None
MODEL_PATH = 'model_artifacts/xgboost_pipeline.joblib'

@app.on_event('startup')
async def load_models():
    global model_pipeline, gemini_model, pest_model
    if not os.path.exists(MODEL_PATH):
        raise RuntimeError(f"Model file not found at {MODEL_PATH}.")
    model_pipeline = joblib.load(MODEL_PATH)
    gemini_model = genai.GenerativeModel('gemini-pro')
    pest_model = MobileNetV2(weights='imagenet')
    print("All models loaded successfully.")

# --- 4. API Endpoints ---
@app.get('/')
async def read_root():
    return {'status': 'active'}

@app.post('/predict')
async def predict(request_data: List[CropYieldRequest]):
    if model_pipeline is None: raise HTTPException(status_code=500, detail="Model not loaded")
    df = pd.DataFrame([item.model_dump() for item in request_data])
    log_predictions = model_pipeline.predict(df)
    return {'predictions': np.expm1(log_predictions).tolist()}

@app.post('/chat')
async def chat(request: ChatRequest):
    if not gemini_model:
        raise HTTPException(status_code=500, detail="Gemini API not configured")
    response = await gemini_model.generate_content_async(request.message)
    return {'response': response.text}

@app.post('/detect_pest')
async def detect_pest_endpoint(file: UploadFile = File(...)):
    image_bytes = await file.read()
    img = Image.open(io.BytesIO(image_bytes)).resize((224, 224))
    x = preprocess_input(np.expand_dims(image.img_to_array(img), axis=0))
    preds = decode_predictions(pest_model.predict(x), top=3)[0]
    return {"detections": [{"label": l, "description": d, "probability": float(p)} for (_, l, p) in preds]}

Writing main.py


In [23]:
import joblib
import os

print(f"'joblib' library is installed and located at: {os.path.dirname(joblib.__file__)}")
print(f"Version of joblib: {joblib.__version__}")

# You can also use help(joblib) to see its documentation.
# help(joblib)


'joblib' library is installed and located at: /usr/local/lib/python3.12/dist-packages/joblib
Version of joblib: 1.5.3


In [24]:
import streamlit as st
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions

@st.cache_resource
def load_pest_detection_model():
    try:
        # Ensure weights are downloaded or available
        model = MobileNetV2(weights='imagenet')
        return model
    except Exception as e:
        st.error(f"Error loading Pest Detection model: {e}")
        return None

pest_detection_model = load_pest_detection_model()

2026-03-31 23:47:49.643 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [25]:
!pip install -q -U google-generativeai google-cloud-aiplatform

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 92.7 MB/s eta 0:00:00


In [28]:
import pandas as pd

# Create a sample market and trends dataset
market_data = {
    'Crop': ['Wheat', 'Maize', 'Potatoes', 'Soybeans'],
    'Current_Price_hg_ha': [450, 320, 150, 600],
    'Trend': ['Upward', 'Stable', 'Downward', 'Upward'],
    'Season': ['Winter', 'Summer', 'All', 'Summer']
}

pd.DataFrame(market_data).to_csv('market_trends.csv', index=False)
print("Market trends data created.")

Market trends data created.


In [27]:
%%writefile streamlit_app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os
import io
from PIL import Image
import asyncio

# Imports for Gemini API
import google.generativeai as genai
from google.colab import userdata

# Imports for Pest Detection
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image

# --- Configuration ---
MODEL_PATH = 'model_artifacts/xgboost_pipeline.joblib'
MARKET_PATH = 'market_trends.csv'

@st.cache_resource
def load_models():
    yield_model = joblib.load(MODEL_PATH) if os.path.exists(MODEL_PATH) else None
    pest_model = MobileNetV2(weights='imagenet')
    api_key = userdata.get('yield_key')
    genai.configure(api_key=api_key)
    gemini = genai.GenerativeModel('gemini-pro')
    return yield_model, pest_model, gemini

yield_pipeline, pest_model, gemini_model = load_models()

st.set_page_config(page_title='Farmer Advisor & Market', layout='wide')
st.title('🌾 Crop Advisor & Marketplace')

tabs = st.tabs(['📈 Yield Prediction', '🪲 Pest Detection', '🤖 AI Advisor', '🛒 Market & Trends'])

with tabs[0]:
    st.header('Yield Prediction')
    with st.form('yield_form'):
        area = st.selectbox('Area', ['Zambia', 'Zimbabwe'])
        item = st.selectbox('Crop', ['Wheat', 'Maize', 'Potatoes', 'Rice, paddy', 'Sorghum', 'Soybeans'])
        year = st.number_input('Year', 2024, 2030, 2025)
        rain = st.slider('Rainfall (mm)', 0, 3500, 1000)
        pest = st.slider('Pesticides (tonnes)', 0, 150000, 5000)
        temp = st.slider('Temp (°C)', 10, 45, 25)
        if st.form_submit_button('Predict'):
            input_df = pd.DataFrame([[area, item, year, rain, pest, temp]], columns=['Area', 'Item', 'Year', 'rainfall', 'pesticides', 'temperature'])
            pred = np.expm1(yield_pipeline.predict(input_df))[0]
            st.metric('Estimated Yield', f'{pred:,.2f} hg/ha')

with tabs[1]:
    st.header('Pest Identification')
    file = st.file_uploader('Upload leaf or pest image', type=['jpg', 'png'])
    if file:
        img = Image.open(file).resize((224, 224))
        st.image(img)
        x = preprocess_input(np.expand_dims(image.img_to_array(img), axis=0))
        preds = decode_predictions(pest_model.predict(x), top=3)[0]
        for _, label, prob in preds:
            st.write(f'**{label}**: {prob:.2%}')

with tabs[2]:
    st.header('AI Chatbot')
    if 'messages' not in st.session_state: st.session_state.messages = []
    for m in st.session_state.messages: st.chat_message(m['role']).write(m['content'])
    if prompt := st.chat_input('How can I improve my soil?'):
        st.session_state.messages.append({'role': 'user', 'content': prompt})
        st.chat_message('user').write(prompt)
        response = gemini_model.generate_content(prompt)
        st.session_state.messages.append({'role': 'assistant', 'content': response.text})
        st.chat_message('assistant').write(response.text)

with tabs[3]:
    st.header('🛒 Farmer Marketplace & Trending Crops')
    col1, col2 = st.columns(2)
    with col1:
        st.subheader('Trending Crops This Season')
        if os.path.exists(MARKET_PATH):
            df_trends = pd.read_csv(MARKET_PATH)
            st.dataframe(df_trends, use_container_width=True)
        else: st.info('No trend data available.')
    with col2:
        st.subheader('List Your Crop for Sale')
        with st.form('market_form'):
            seller_name = st.text_input('Name')
            crop_type = st.selectbox('Crop', ['Wheat', 'Maize', 'Potatoes', 'Rice, paddy', 'Sorghum', 'Soybeans'])
            quantity = st.number_input('Quantity (kg)', min_value=1)
            price = st.number_input('Asking Price ($)', min_value=1)
            if st.form_submit_button('Post Listing'):
                st.success(f'Listing created for {seller_name}! Others can now see your {crop_type}.')
                # In a real app, this would append to a database/CSV

Writing streamlit_app.py


In [29]:
%%writefile requirements.txt
streamlit
pandas
numpy
joblib
scikit-learn
xgboost
tensorflow-cpu
Pillow
google-generativeai

Writing requirements.txt
